In [1]:
import requests
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import numpy as np
import re

res = requests.get('https://registrar.ucdavis.edu/calendar/archive/academic-calendar')

# Using pd.read_html() instead

In [2]:
archivedYears = pd.read_html(
    'https://registrar.ucdavis.edu/calendar/archive/academic-calendar',
    storage_options={'User-Agent':'Mozilla/5.0'}
    )

In [3]:
currentYears = pd.read_html(
    'https://registrar.ucdavis.edu/calendar/academic-calendar',
    storage_options={'User-Agent':'Mozilla/5.0'}
    )

In [4]:
academicYears = archivedYears + currentYears

### Helper functions

In [5]:
def getInstructionDateRange(quarterDates, year):
    # Find quarter begin and end as a string
    instructionBeginStr = quarterDates[quarterDates["Desc"] == 'Instruction Begins']["Dates"].array[0]
    instructionEndsStr = quarterDates[quarterDates["Desc"] == 'Instruction Ends']["Dates"].array[0]
    
    # Add year to the string
    instructionBeginStr = instructionBeginStr + '/' + str(year)
    instructionEndsStr = instructionEndsStr + '/' + str(year)
    
    # Create datetime objects using the strings
    instructionBeginDateTime = datetime.datetime.strptime(instructionBeginStr,'%-m/%d/%Y')
    instructionEndsDateTime = datetime.datetime.strptime(instructionEndsStr,'%-m/%d/%Y')

    # Create date range using datetime objects
    instructionRange = pd.date_range(start=instructionBeginDateTime, end=instructionEndsDateTime)
    return instructionRange

Although the UC Davis calendar treats weekends not as finals, we will treat the weekends as essentially still finals because students will still be studying

In [6]:
def getFinalsDateRange(quarterDates, year):
    # Find finals begin/end as a string
    finalsStr = quarterDates[quarterDates["Desc"] == 'Finals Begin/End']["Dates"].array[0]

    # Parse month and day information based on their consistent format
    if finalsStr.find(',') != -1:
        month = int(finalsStr.split('/')[0])
        firstDay = int(finalsStr.split('/')[1].split(',')[0])
        lastDay = int(finalsStr.split('-')[1])
    else:
        month = int(finalsStr.split('/')[0])
        firstDay = int(finalsStr.split('/')[1].split('-')[0])
        lastDay = int(finalsStr.split('-')[1])

    # Create finals start and end datetime objects
    finalsStartDateTime = datetime.datetime(year=year,month=month,day=firstDay)
    finalsEndDateTime = datetime.datetime(year=year,month=month,day=lastDay)

    # Create date range using datetime objects
    finalsRange = pd.date_range(start=finalsStartDateTime, end=finalsEndDateTime)
    return finalsRange

In [7]:
def getHolidaysDateRange(quarterDates, year):
    # Initialize output
    outputDateRange = None

    # Find holidays in quarter dates
    holidays = quarterDates[quarterDates["Desc"].str.contains("Holiday")]
    for _, data in holidays.iterrows():
        dateStr = str(data["Dates"])

        # Initialize some values so that they can be accessed outside of the conditional statements
        firstDateStr = ""
        secondDateStr = ""
        firstYear = year
        secondYear = year
        
        # Find day and month of first day
        # Multiple days case
        if (dateStr.find("&") != -1) or (dateStr.find(",") != -1) or (dateStr.find("-") != -1):
            # Two dates split using '&'
            if len(dateStr.split("&")) == 2:
                firstDateStr, secondDateStr = dateStr.replace(" ","").split("&")
            # Two dates split using ','
            elif len(dateStr.split(",")) == 2:
                firstDateStr, secondDateStr = dateStr.replace(" ","").split(",")
            # Two contiguous dates denoted with '-'
            elif len(dateStr.split("-")) == 2:
                firstDateStr, secondDateStr = dateStr.replace(" ","").split("-")
            firstMonth = int(firstDateStr.split("/")[0])
            firstDay = int(firstDateStr.split("/")[1])
        # Just one day case
        else:
            firstMonth = int(dateStr.split("/")[0])
            firstDay = int(dateStr.split("/")[1])

        # Find day and month of second day
        # Second date is in a different month
        if len(secondDateStr.split("/")) == 2:
            secondMonth = int(secondDateStr.split("/")[0])
            secondDay = int(secondDateStr.split("/")[1])
            # Update secondYear if necessory
            if firstMonth > secondMonth:
                secondYear += 1
        # Second date isn't in a different month
        elif secondDateStr != "":
            secondMonth = firstMonth
            secondDay = int(secondDateStr)
        # No second date at all
        else:
            secondMonth = firstMonth
            secondDay = firstDay        

        # Convert to datetime objects
        firstDateTime = datetime.datetime(firstYear, firstMonth, firstDay)
        secondDateTime = datetime.datetime(secondYear, secondMonth, secondDay)
        
        # Create or append to output datetime range
        if outputDateRange is None:
            outputDateRange = pd.date_range(firstDateTime, secondDateTime)
        else:
            outputDateRange = outputDateRange.union(pd.date_range(firstDateTime, secondDateTime))

    return outputDateRange

### Main date-finding function for a full academic year

In [8]:
def parseAcademicYear(table):
    # Rename columns so it isn't obnoxious to look at
    table = table.rename(columns={
        table.columns[0]:'Desc',
        table.columns[2]:'Dates'
    })
    
    # Find the start and end year of each academic year (each represented by one table)
    startYear = 0
    try:
        startYear = int(re.findall(r'\d+', table.columns[1])[0])
    except:
        startYear = int(re.findall(r'\d+', table[1][0])[0])
        print("Format is too different for {startYear}, skipping this one".format(startYear=startYear))
        return None, None, None, None
    endYear = startYear + 1
    
    # Identify which quarter each row is referring to
    table["Quarter Number"] = table['Desc'].isnull().cumsum()
    
    fallDates = table[table["Quarter Number"] == 0]
    winterDates = table[table["Quarter Number"] == 1]
    springDates = table[table["Quarter Number"] == 2]
    summerDates = table[table["Quarter Number"] == 3]
    
    # Drop second column, doesn't give any really useful information
    # Exception is summer dates, where the format changed in 2023-2024
    # Instead, we drop the null column and rename second column to "Dates" again just in case
    fallDates = fallDates.drop(columns=fallDates.columns[1])
    winterDates = winterDates.drop(columns=winterDates.columns[1])
    springDates = springDates.drop(columns=springDates.columns[1])
    
    if summerDates["Dates"].isna().all():
        summerDates = summerDates.drop(columns="Dates")
        summerDates = summerDates.rename(columns=
                                     {summerDates.columns[1]:'Dates'}
                                      )
    else:
        summerDates = summerDates.drop(columns=summerDates.columns[1])

    fallInstruction = getInstructionDateRange(fallDates, startYear)
    winterInstruction = getInstructionDateRange(winterDates, endYear)
    springInstruction = getInstructionDateRange(springDates, endYear)
    instructionRange = fallInstruction.union(winterInstruction).union(springInstruction)

    fallFinals = getFinalsDateRange(fallDates, startYear)
    winterFinals = getFinalsDateRange(winterDates, endYear)
    springFinals = getFinalsDateRange(springDates, endYear)
    finalsRange = fallFinals.union(winterFinals).union(springFinals)

    fallHolidays = getHolidaysDateRange(fallDates, startYear)
    winterHolidays = getHolidaysDateRange(winterDates, endYear)
    springHolidays = getHolidaysDateRange(springDates, endYear)
    summerHolidays = getHolidaysDateRange(summerDates, endYear)
    holidaysRange = fallHolidays.union(winterHolidays).union(springHolidays).union(summerHolidays)

    return instructionRange, finalsRange, holidaysRange, startYear

In [9]:
# Initialize outputs
fullInstructionRange = None
fullFinalsRange = None
fullHolidaysRange = None

# Iterate through archived years
for table in academicYears:
    # Gather info from the academic year table
    instructionRange, finalsRange, holidaysRange, startYear = parseAcademicYear(table)

    # Skip iteration if information can't be parsed
    if instructionRange is None:
        continue

    # Construct output
    if fullInstructionRange is None:
        fullInstructionRange = instructionRange
        fullFinalsRange = finalsRange
        fullHolidaysRange = holidaysRange
    else:
        fullInstructionRange = fullInstructionRange.union(instructionRange)
        fullFinalsRange = fullFinalsRange.union(finalsRange)  
        fullHolidaysRange = fullHolidaysRange.union(holidaysRange)
    
    # print(startYear)
    # print(instructionRange)
    # print(finalsRange)
    # print(holidaysRange)

display(fullInstructionRange, fullFinalsRange, fullHolidaysRange)

Format is too different for 2017, skipping this one
Format is too different for 2016, skipping this one


DatetimeIndex(['2018-09-26', '2018-09-27', '2018-09-28', '2018-09-29',
               '2018-09-30', '2018-10-01', '2018-10-02', '2018-10-03',
               '2018-10-04', '2018-10-05',
               ...
               '2027-05-25', '2027-05-26', '2027-05-27', '2027-05-28',
               '2027-05-29', '2027-05-30', '2027-05-31', '2027-06-01',
               '2027-06-02', '2027-06-03'],
              dtype='datetime64[us]', length=1872, freq=None)

DatetimeIndex(['2018-12-10', '2018-12-11', '2018-12-12', '2018-12-13',
               '2018-12-14', '2019-03-18', '2019-03-19', '2019-03-20',
               '2019-03-21', '2019-03-22',
               ...
               '2027-03-17', '2027-03-18', '2027-03-19', '2027-06-04',
               '2027-06-05', '2027-06-06', '2027-06-07', '2027-06-08',
               '2027-06-09', '2027-06-10'],
              dtype='datetime64[us]', length=153, freq=None)

DatetimeIndex(['2018-11-12', '2018-11-22', '2018-11-23', '2018-12-24',
               '2018-12-25', '2018-12-31', '2019-01-01', '2019-01-21',
               '2019-02-18', '2019-03-29',
               ...
               '2026-12-25', '2026-12-31', '2027-01-01', '2027-01-18',
               '2027-02-15', '2027-03-26', '2027-05-31', '2027-06-18',
               '2027-07-05', '2027-09-06'],
              dtype='datetime64[us]', length=130, freq=None)

In [10]:
# Export these date ranges
pd.to_pickle(fullInstructionRange, '../data/date pkl/instructionDates.pkl')
pd.to_pickle(fullFinalsRange, '../data/date pkl/finalsDates.pkl')
pd.to_pickle(fullHolidaysRange, '../data/date pkl/holidayDates.pkl')